In [1]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

# SAM 3 Agent

This notebook shows an example of how an MLLM can use SAM 3 as a tool, i.e., "SAM 3 Agent", to segment more complex text queries such as "the leftmost child wearing blue vest".

## Env Setup

First install `sam3` in your environment using the [installation instructions](https://github.com/facebookresearch/sam3?tab=readme-ov-file#installation) in the repository.

In [2]:
%pip list | grep sam3

In [3]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [4]:
import os

SAM3_ROOT = os.path.dirname(os.getcwd())
os.chdir(SAM3_ROOT)

# setup GPU to use -  A single GPU is good with the purpose of this demo
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
_ = os.system("nvidia-smi")

## Build SAM3 Model

In [5]:
%ls sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz



In [6]:
import sam3
from sam3.sam3 import build_sam3_image_model
from sam3.sam3.model.sam3_image_processor import Sam3Processor

# sam3_root = os.path.dirname(sam3.__file__)
bpe_path = f"sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)
processor = Sam3Processor(model, confidence_threshold=0.5)

## LLM Setup

Config which MLLM to use, it can either be a model served by vLLM that you launch from your own machine or a model is served via external API. If you want to using a vLLM model, we also provided insturctions below.

In [8]:
import os
LLM_CONFIGS = {
    # vLLM-served models
    "qwen3_vl_8b_thinking": {
        "provider": "vllm",
        "model": "Qwen/Qwen3-VL-8B-Thinking",
    },
    "gpt-5.2": {
        "provider": "openai",
        "model": "gpt-5.2",
        "base_url": "https://api.openai.com/v1",
        "api_key": os.getenv("OPENAI_API_KEY"),
    },
    # models served via external APIs
    # add your own
}

# model = "qwen3_vl_8b_thinking"
model = "gpt-5.2"
LLM_API_KEY = os.getenv("OPENAI_API_KEY")

llm_config = LLM_CONFIGS[model]
# llm_config["api_key"] = LLM_API_KEY
llm_config["name"] = model

# setup API endpoint
if llm_config["provider"] == "vllm":
    LLM_SERVER_URL = "http://127.0.0.1:8002/v1"  # replace this with your vLLM server address as needed
else:
    LLM_SERVER_URL = llm_config["base_url"]

### Setup vLLM server 
This step is only required if you are using a model served by vLLM, skip this step if you are calling LLM using an API like Gemini and GPT.

* Install vLLM (in a separate conda env from SAM 3 to avoid dependency conflicts).
  ```bash
    conda create -n vllm python=3.12
    pip install vllm --extra-index-url https://download.pytorch.org/whl/cu128
  ```
* Start vLLM server on the same machine of this notebook
  ```bash
    # qwen 3 VL 8B thinking
    vllm serve Qwen/Qwen3-VL-8B-Thinking --tensor-parallel-size 4 --allowed-local-media-path / --enforce-eager --port 8001
  ```

## Run SAM3 Agent Inference

In [9]:
from functools import partial
from IPython.display import display, Image
from sam3.agent.client_llm import send_generate_request as send_generate_request_orig
from sam3.agent.client_sam3 import call_sam_service as call_sam_service_orig
from sam3.agent.inference import run_single_image_inference

In [ ]:
%pwd

In [10]:
video_path = "sam3/vids/chinup_final.mp4"
import cv2
from PIL import Image
cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
frame.shape
img = Image.fromarray(frame)
img.save("sam3/examples/test_image.jpg")

In [ ]:
# prepare input args and run single image inference
image = "sam3/examples/test_image.jpg"
user_prompt = """
I want to detect the hands in the video that is holding the barbell.
"""
image = os.path.abspath(image)
send_generate_request = partial(send_generate_request_orig, server_url=LLM_SERVER_URL, model=llm_config["model"], api_key=llm_config["api_key"])
call_sam_service = partial(call_sam_service_orig, sam3_processor=processor)
# todo: add prediction probability as information
run_single_image_inference(
    image, user_prompt, llm_config, send_generate_request, call_sam_service,
    debug=True, output_dir="tracker_output"
)

# display output
# if output_image_path is not None:
#     display(Image(filename=output_image_path))

In [16]:
from sam3.sam3.agent.agent_video_tracker import Sam3TrackingTool
sam3_tracker = Sam3TrackingTool(video_path, bpe_path)

In [ ]:
detection = sam3_tracker._add_prompt("hands")
detection

In [15]:
import json
with open("/root/thomas_workspace/tracker_output/sam_out/-root-thomas_workspace-sam3-examples-test_image.jpg/hands.json", "r") as f:
    data = json.load(f)
data


In [ ]:
# todo: add native support
